# k-Nearest Neighbour para decidir cuál modelo es mejor para cada consulta

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import joblib
import gower

In [ ]:
data = "resultados.json"
data = pd.read_json(data)

In [ ]:
print(data.head())

In [ ]:
data.columms()

In [ ]:
# Crear el DataFrame con las variables predictoras
datos = data.drop(columns=['model','usuario', 'response_preview', 'query', 'status_code'])  # Suponiendo que 'model' es la columna objetivo
datos['target'] = data['model']

display(datos.head())  

# Mostrar número de ejemplos y variables
print(f"Número de ejemplos (filas): {datos.shape[0]}")
print(f"Número de variables totales (columnas): {datos.shape[1]}")

# 1. Seleccionamos solo las columnas numéricas
columnas_numericas = datos.select_dtypes(include=[np.number]).columns

# 2. Calculamos las medianas de esas columnas
medianas = datos[columnas_numericas].median()

# 3. Rellenamos los nulos de esas columnas con sus respectivas medianas
datos[columnas_numericas] = datos[columnas_numericas].fillna(medianas)

print("¡Nulos imputados con la mediana en las variables numéricas!")

# 1. Identificamos las columnas que tienen el tipo de datos conflictivo (StringDtype)
columnas_string_conflictivas = datos.select_dtypes(include=['string']).columns

# 2. Las convertimos al tipo nativo 'object' que Scikit-Learn sí entiende
for col in columnas_string_conflictivas:
    datos[col] = datos[col].astype(object)

print("¡Conversión de tipos completada con éxito!")

In [ ]:
# Separamos características (X) y objetivo (y)
y = datos['target']
X = datos.drop(columns=['target'])

# Dividimos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y
)

# Comprobamos la distribución (en porcentaje)
print("Distribución en Entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en Prueba:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
X_train_dist = gower.gower_matrix(X_train)

# Inicializamos el modelo k-NN con k=3 y distancia de Gower
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')

from sklearn.preprocessing import OrdinalEncoder

# 1. Identificamos cuáles son las columnas de texto/categóricas
columnas_categoricas = datos.select_dtypes(include=['object', 'string']).columns

# (Asegúrate de calcular tu 'cat_mask' y tus 'rangos' basándote en el 
# DataFrame original ANTES de aplicar la transformación del siguiente paso).

# 2. Creamos una copia de los datos para no machacar los originales
datos_numericos = datos.copy()

# 3. Transformamos las palabras en números enteros (ej: 'marketing' -> 0, 'ventas' -> 1)
codificador = OrdinalEncoder()
datos_numericos[columnas_categoricas] = codificador.fit_transform(datos[columnas_categoricas])

# 1. Calculamos la matriz cuadrada (180x180) de entrenamiento
matriz_distancias_train = gower.gower_matrix(X_train) 

# 2. Instanciamos el modelo ESPERANDO esa matriz cuadrada
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')

# 3. CORRECTO: Le pasamos la matriz de distancias (180x180), no los datos originales
knn.fit(matriz_distancias_train, y_train)

print("¡Modelo entrenado superando el bloqueo de texto!")

In [ ]:
X_test_dist = gower.gower_matrix(X_test, X_train)

# Generamos las predicciones sobre el conjunto de prueba
y_pred = knn.predict(X_test_dist)

print("Matriz de entrenamiento mapeada con éxito.")
print(f"Predicciones del conjunto de prueba: {y_pred}")

# Contamos cuántas predicciones coinciden exactamente con la realidad
aciertos = (y_test == y_pred).sum()
total_casos = len(y_test)

print(f"El modelo ha acertado {aciertos} de un total de {total_casos} casos.")

In [ ]:
joblib.dump(knn, 'modelo_knn_ej8.joblib')
print("¡Modelo exportado correctamente!")